# Möbius Galton Board Explorer

Interactive Jupyter notebook for exploring the Möbius Galton Board simulations.

**Key formula:**  $\alpha_+(W, H) = \frac{1}{2}\left[1 + \cos^H\!\left(\frac{\pi}{W}\right)\right]$

**Phase transition at:** $k^* = \frac{1}{2\ln 2} \approx 0.7213$

**Critical width:** $W_e \approx 2.46\sqrt{H}$

**Inverse formula:** $W = \dfrac{\pi}{\arccos\!\left[(2\alpha_+ - 1)^{1/H}\right]}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
%matplotlib inline

K_STAR = 1.0 / (2.0 * np.log(2.0))
print(f'k* = {K_STAR:.6f}')

## 1. Analytic Formula

In [ ]:
def alpha_plus(W: float, H: int) -> float:
    return 0.5 * (1.0 + np.cos(np.pi / W) ** H)

def inverse_W(alpha: float, H: int):
    if not (0.5 < alpha < 1.0): return None
    inner = (2.0 * alpha - 1.0) ** (1.0 / H)
    if abs(inner) > 1: return None
    return float(np.pi / np.arccos(inner))

def critical_width(H: int) -> float:
    return inverse_W(K_STAR, H)

## 2. Interactive α₊ vs W Plot

In [ ]:
def plot_alpha(H=25, W_max=35):
    W_vals = np.linspace(2.01, W_max, 500)
    a_vals = [alpha_plus(w, H) for w in W_vals]
    Wc = critical_width(H)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(W_vals, a_vals, 'b-', lw=2, label=f'α₊(W, H={H})')
    ax.axhline(K_STAR, color='gold',   ls='--', lw=1.5, label=f'k* = {K_STAR:.4f}')
    ax.axvline(Wc,     color='purple', ls=':',  lw=1.5, label=f'Wₑ = {Wc:.2f}')
    ax.axhline(0.5,    color='gray',   ls=':',  lw=1.0, alpha=0.5)
    ax.fill_between(W_vals, K_STAR, a_vals,
                    where=np.array(a_vals) > K_STAR,
                    alpha=0.12, color='blue', label='α₊ > k* region')
    ax.set_xlabel('Width W')
    ax.set_ylabel('α₊')
    ax.set_title(f'Möbius Galton Board Phase Transition  (H={H})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.45, 1.02)
    plt.tight_layout()
    plt.show()
    print(f'Wₑ = {Wc:.4f}   (≈ 2.46·√{H} = {2.46*H**0.5:.4f})')

interact(plot_alpha,
    H=IntSlider(min=5, max=100, step=1, value=25, description='H'),
    W_max=IntSlider(min=10, max=60, step=1, value=35, description='W max'))

## 3. Monte Carlo Simulation

In [ ]:
def run_mc(W, H, N=100_000, mode='mobius', seed=42):
    rng = np.random.default_rng(seed)
    positions = np.zeros(N, dtype=np.float64)
    parities  = np.zeros(N, dtype=np.int32)
    for _ in range(H):
        steps = rng.integers(0, 2, size=N) * 2 - 1
        positions += steps
        if mode == 'standard':
            positions = np.where(positions < 0, -positions, positions)
            positions = np.where(positions >= W, 2*W-2-positions, positions)
        elif mode == 'cylinder':
            positions = positions % W
        elif mode == 'mobius':
            crossings = (positions // W).astype(np.int32)
            positions = positions % W
            parities  = (parities + np.abs(crossings)) % 2
    return float(np.mean(parities == 0)), positions.astype(int), parities

def interactive_mc(W=14, H=25, N=50000, mode='mobius'):
    a_mc, positions, parities = run_mc(W, H, N, mode)
    a_th = alpha_plus(W, H) if mode == 'mobius' else None
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    ax = axes[0]
    bins = np.arange(W+1) - 0.5
    even_pos = positions[parities == 0]
    odd_pos  = positions[parities == 1]
    ax.hist(even_pos, bins=bins, alpha=0.7, color='steelblue',
            label=f'Even  α₊={a_mc:.4f}', edgecolor='white')
    ax.hist(odd_pos,  bins=bins, alpha=0.7, color='tomato',
            label='Odd', edgecolor='white')
    ax.set_xlabel('Final position')
    ax.set_ylabel('Count')
    ax.set_title(f'{mode.capitalize()} board (W={W}, H={H})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # α₊ gauge
    ax2 = axes[1]
    ax2.barh(['Simulation'], [a_mc],  color='steelblue', alpha=0.8, label=f'MC: {a_mc:.4f}')
    if a_th is not None:
        ax2.barh(['Theory'],    [a_th],  color='green',     alpha=0.7, label=f'Theory: {a_th:.4f}')
    ax2.axvline(K_STAR, color='gold', ls='--', lw=2, label=f'k*={K_STAR:.4f}')
    ax2.axvline(0.5,    color='gray', ls=':',  lw=1)
    ax2.set_xlim(0.4, 1.05)
    ax2.set_xlabel('α₊')
    ax2.set_title('Even-parity fraction')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    if a_th: print(f'|MC − theory| = {abs(a_mc - a_th):.5f}')

interact(interactive_mc,
    W=IntSlider(min=4, max=30, step=1, value=14, description='W'),
    H=IntSlider(min=5, max=60, step=1, value=25, description='H'),
    N=IntSlider(min=5000, max=200000, step=5000, value=50000, description='N'),
    mode=Dropdown(options=['standard','cylinder','mobius'], value='mobius', description='mode'))

## 4. Inverse Solver

In [ ]:
def interactive_inverse(alpha=K_STAR, H=25):
    W_rec = inverse_W(alpha, H)
    Wc    = critical_width(H)
    print(f'Input:  α₊ = {alpha:.6f},  H = {H}')
    print(f'Output: W  = {W_rec:.6f}' if W_rec else 'Output: α₊ out of range (must be in (0.5, 1))')
    if W_rec:
        rt = alpha_plus(W_rec, H)
        print(f'Round-trip α₊ = {rt:.8f}  (error = {abs(rt - alpha):.2e})')
    print(f'Critical width Wₑ(H={H}) = {Wc:.4f}  ≈ 2.46·√{H} = {2.46*H**0.5:.4f}')

interact(interactive_inverse,
    alpha=FloatSlider(min=0.501, max=0.999, step=0.001, value=K_STAR,
                      description='α₊', readout_format='.4f'),
    H=IntSlider(min=5, max=100, step=1, value=25, description='H'))

## 5. Phase Diagram: α₊(W, H)

In [ ]:
W_grid = np.linspace(3, 40, 200)
H_grid = np.arange(5, 81, 1)
alpha_grid = np.array([[alpha_plus(w, h) for w in W_grid] for h in H_grid])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.contourf(W_grid, H_grid, alpha_grid, levels=50, cmap='plasma')
plt.colorbar(im, ax=ax, label='α₊')

# k* contour
cs = ax.contour(W_grid, H_grid, alpha_grid, levels=[K_STAR],
                colors='yellow', linewidths=2)
ax.clabel(cs, fmt=f'k*={K_STAR:.3f}', fontsize=10)

# Wₑ ≈ 2.46√H curve
H_curve = np.linspace(5, 80, 200)
W_curve = 2.46 * H_curve**0.5
ax.plot(W_curve, H_curve, 'w--', lw=1.5, label='Wₑ ≈ 2.46√H')

ax.set_xlabel('Width W')
ax.set_ylabel('Height H')
ax.set_title('Phase diagram: α₊(W, H)  [Möbius board]')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 6. Validation Table (H=25)

In [ ]:
H = 25
print(f'Validation table  H={H}')
print(f'{"W":>4}  {"α₊ (theory)":>14}  {"α₊ (MC, N=100k)":>18}')
print('-' * 42)
for W in range(6, 31, 2):
    a_th = alpha_plus(W, H)
    a_mc, _, _ = run_mc(W, H, N=100_000)
    print(f'{W:>4}  {a_th:>14.5f}  {a_mc:>18.5f}')